In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from config.load_config import load_config

from models.factory import ModelFactory
from core.evaluator import Evaluator

import numpy as np

from pathlib import Path
import joblib
import json

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import numpy as np

from core.data_loader import DataLoader

In [2]:
config = load_config("config/config.yaml")
data_loader = DataLoader(config)

In [3]:
df = pd.read_csv(config["data"]["path"]).dropna(subset=["Precipitation(in)"])
date_format = config["features"]["date_format"]
dates = pd.to_datetime(df[config["features"]["temporal"]], format=date_format)
for feature in config["features"]["temporal_features"]:
  df[feature] = getattr(dates.dt, feature)
  df[feature] = df[feature].astype(float)
df[config["features"]["categorical"]] = df[config["features"]["categorical"]].astype("string")
df["hour"] = df["hour"] + df["minute"] / 60
df.drop(columns=[config["features"]["temporal"]], inplace=True)

In [4]:
df[config['features']["categorical"]] = (
    df[config['features']["categorical"]]
    .astype(object)
    .fillna("__missing__")
)

In [5]:
X_train, X_val, X_test, y_train, y_val, y_test = data_loader.split(df)

In [6]:
temporal_features = [c for c in config['features']["temporal_features"] if c != "minute"]
periods = {"hour": 24, "month": 12, "dayofweek": 7}

def transform(df: pd.DataFrame):
  df = df.copy()

  for feature in temporal_features:
    period = periods[feature]
    df[f"{feature}_sin"] = np.sin(2 * np.pi * df[feature] / period)
    df[f"{feature}_cos"] = np.cos(2 * np.pi * df[feature] / period)

  df.drop(columns=temporal_features, inplace=True, errors="ignore")
  return df

In [7]:
X_train = transform(X_train)
X_val = transform(X_val)
X_test = transform(X_test)

In [8]:
numerical = (c for c in config["features"]["numerical"] if c != "Precipitation(in)")
preprocessor = ColumnTransformer([
  ("num", Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
    ]), numerical),
  ("precip", Pipeline([
    ("log", FunctionTransformer(np.log1p)),
    ("scaler", StandardScaler())
  ]), ["Precipitation(in)"]),
  ("cat", Pipeline([
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
  ]), config["features"]["categorical"]),
])

In [9]:
preprocessor.fit(X_train)

ValueError: No valid specification of the columns. Only a scalar, list or slice of all integers or all strings, or boolean mask is allowed

In [ ]:
X_train = preprocessor.transform(X_train)

In [ ]:
X_val = preprocessor.transform(X_val)

In [ ]:
X_test = preprocessor.transform(X_test)

In [10]:
df.isna().mean().sort_values(ascending=False)


End_Lng                  3.301934e-01
End_Lat                  3.301934e-01
Wind_Chill(F)            4.530981e-02
Wind_Speed(mph)          1.446421e-02
Wind_Direction           9.629837e-03
Humidity(%)              6.346827e-03
Visibility(mi)           6.102113e-03
Temperature(F)           4.845236e-03
Nautical_Twilight        3.671259e-03
Astronomical_Twilight    3.671259e-03
Sunrise_Sunset           3.671259e-03
Civil_Twilight           3.671259e-03
Street                   1.866128e-03
Pressure(in)             1.858526e-03
City                     3.547635e-05
Description              7.240071e-07
ID                       0.000000e+00
Timezone                 0.000000e+00
Country                  0.000000e+00
Zipcode                  0.000000e+00
State                    0.000000e+00
County                   0.000000e+00
End_Time                 0.000000e+00
Start_Lng                0.000000e+00
Distance(mi)             0.000000e+00
Severity                 0.000000e+00
Source      

In [ ]:
df["Severity"].isna().mean()


np.float64(0.0)

: 

In [ ]:
mask = df['Weather_Condition'].apply(lambda x: isinstance(x, float))
df1 = df.loc[mask, 'Weather_Condition']

In [ ]:
df['Weather_Condition'].apply(lambda x: isinstance(x, float)).sum()
